In [1]:
from datetime import datetime
from zoneinfo import ZoneInfo
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook"

In [2]:
ist_now = datetime.now(ZoneInfo("Asia/Kolkata"))
current_hour = ist_now.hour

In [ ]:
np.random.seed(42)

categories       = ['GAME','TOOLS','ENTERTAINMENT','PRODUCTIVITY','FAMILY',
                    'COMMUNICATION','SOCIAL','PHOTOGRAPHY','FINANCE','EDUCATION']
app_types        = ['Free','Paid']
content_ratings  = ['Everyone','Teen','Mature 17+','Everyone 10+']
android_versions = ['2.3','3.0','4.0','4.1','4.2','4.4','5.0','6.0','7.0','8.0']

def random_app_name():
    words = ['Pro','Plus','App','Go','Lite','HD','Free','Smart','Fast','Easy',
             'Quick','Super','Mega','Top','Best','My','The','New','Advanced','Simple']
    return (' '.join(np.random.choice(words, size=np.random.randint(1,4))))[:40]

rows = []
for _ in range(3000):
    cat      = np.random.choice(categories,
                                p=[0.15,0.12,0.11,0.10,0.12,0.09,0.08,0.08,0.08,0.07])
    atype    = np.random.choice(app_types, p=[0.7, 0.3])
    installs = int(np.random.choice(
                   [500,1000,5000,10000,50000,100000,500000,1000000],
                   p=[0.05,0.08,0.10,0.15,0.20,0.18,0.14,0.10]))
    price    = 0.0 if atype == 'Free' else round(
                   np.random.choice([0.99,1.99,2.99,4.99,9.99,14.99]), 2)
    revenue  = 0   if atype == 'Free' else int(installs * price * np.random.uniform(0.5, 1.5))
    rows.append({
        'App'            : random_app_name(),
        'Category'       : cat,
        'Type'           : atype,
        'Installs'       : installs,
        'Price'          : price,
        'Revenue'        : revenue,
        'Size_MB'        : round(np.random.uniform(1, 100), 1),
        'Android_Ver'    : np.random.choice(android_versions),
        'Content_Rating' : np.random.choice(content_ratings, p=[0.5,0.25,0.15,0.10]),
        'Rating'         : round(np.random.uniform(1.0, 5.0), 1)
    })

apps_df = pd.DataFrame(rows)

# Convert Android version string → float for numeric comparison
# Example: "4.1" → 4.1 so we can do > 4.0 comparison
apps_df['Android_Ver_Float'] = pd.to_numeric(
    apps_df['Android_Ver'], errors='coerce'
)

# Count app name length including spaces and special characters
apps_df['App_Name_Len'] = apps_df['App'].str.len()

print(f"Total rows loaded  : {len(apps_df)}")
print(f"Columns            : {apps_df.columns.tolist()}")
print("\nSample rows:")
apps_df[['App','Category','Type','Installs','Revenue',
         'Size_MB','Android_Ver','Content_Rating','App_Name_Len']].head(5)

Total rows loaded  : 3000
Columns            : ['App', 'Category', 'Type', 'Installs', 'Price', 'Revenue', 'Size_MB', 'Android_Ver', 'Content_Rating', 'Rating', 'Android_Ver_Float', 'App_Name_Len']

Sample rows:


,App,Category,Type,Installs,Revenue,Size_MB,Android_Ver,Content_Rating,App_Name_Len
0,Advanced Quick,ENTERTAINMENT,Paid,100000,1095753,46.5,4.2,Teen,14
1,Pro,GAME,Paid,500000,3753336,31.1,4.4,Everyone,3
2,Best Advanced Super,FAMILY,Free,500,0,51.9,7.0,Everyone,19
3,New Fast,TOOLS,Free,1000000,0,2.6,3.0,Teen,8
4,The Go,GAME,Free,500,0,19.0,4.1,Everyone,6


In [4]:
filtered_df = apps_df[
    (apps_df['Installs']          >= 10000)      &
    (apps_df['Revenue']           >= 10000)      &
    (apps_df['Android_Ver_Float'] >  4.0)        &
    (apps_df['Size_MB']           >  15)         &
    (apps_df['Content_Rating']    == 'Everyone') &
    (apps_df['App_Name_Len']      <= 30)
]

In [ ]:
# Top 3 categories by total installs after filtering
top3_cats = (
    filtered_df.groupby('Category')['Installs']
    .sum()
    .nlargest(3)
    .index
    .tolist()
)
print(f"Top 3 Categories by Total Installs: {top3_cats}")

# ✅ THIS LINE WAS MISSING — filter dataset to only top 3 categories
top3_df = filtered_df[filtered_df['Category'].isin(top3_cats)].copy()

# Aggregate: mean installs and mean revenue per Category per Type (Free/Paid)
agg = top3_df.groupby(['Category', 'Type']).agg(
    Avg_Installs = ('Installs', 'mean'),
    Avg_Revenue  = ('Revenue',  'mean'),
    App_Count    = ('App',      'count')
).reset_index()

# Round values for clean display
agg['Avg_Installs'] = agg['Avg_Installs'].round(0).astype(int)
agg['Avg_Revenue']  = agg['Avg_Revenue'].round(2)

print("\nAggregated Data (used in chart):")
print(agg.to_string(index=False))

Top 3 Categories by Total Installs: [np.str_('FINANCE'), np.str_('GAME'), np.str_('COMMUNICATION')]

Aggregated Data (used in chart):
     Category Type  Avg_Installs  Avg_Revenue  App_Count
COMMUNICATION Paid        325238   2080865.67         21
      FINANCE Paid        352400   3513239.92         25
         GAME Paid        248710   1726618.42         31


In [ ]:
if 13 <= current_hour < 14:

    # ── Prepare Free and Paid dataframes ───────────────────────
    # reindex → ensures all 3 categories appear even if one type is missing
    # fillna(0) → replaces missing values with 0 instead of NaN
    free_df = agg[agg['Type']=='Free'].set_index('Category').reindex(top3_cats).fillna(0)
    paid_df = agg[agg['Type']=='Paid'].set_index('Category').reindex(top3_cats).fillna(0)

    # ── X positions for 4 bars per category ────────────────────
    # Each category gets index 0, 1, 2
    # 4 bars are offset by 0.2 units each to sit side by side
    x    = list(range(len(top3_cats)))
    x_fi = [i - 0.3 for i in x]   # Free  Installs → leftmost bar
    x_pi = [i - 0.1 for i in x]   # Paid  Installs → second bar
    x_fr = [i + 0.1 for i in x]   # Free  Revenue  → third bar
    x_pr = [i + 0.3 for i in x]   # Paid  Revenue  → rightmost bar

    fig = go.Figure()

    # BAR 1 — Free Apps: Avg Installs (Left Y-axis) — Blue
    fig.add_trace(go.Bar(
        name         = 'Free — Avg Installs',
        x            = x_fi,
        y            = free_df['Avg_Installs'].tolist(),
        width        = 0.18,
        marker       = dict(color='#4f9cf9', opacity=0.85,
                            line=dict(color='#4f9cf9', width=1)),
        yaxis        = 'y1',
        text         = [f"{int(v):,}" for v in free_df['Avg_Installs']],
        textposition = 'outside',
        textfont     = dict(color='#4f9cf9', size=9),
        hovertemplate= '<b>Free — Avg Installs</b><br>Installs: %{y:,.0f}<extra></extra>'
    ))

    # BAR 2 — Paid Apps: Avg Installs (Left Y-axis) — Green
    fig.add_trace(go.Bar(
        name         = 'Paid — Avg Installs',
        x            = x_pi,
        y            = paid_df['Avg_Installs'].tolist(),
        width        = 0.18,
        marker       = dict(color='#34d399', opacity=0.85,
                            line=dict(color='#34d399', width=1)),
        yaxis        = 'y1',
        text         = [f"{int(v):,}" for v in paid_df['Avg_Installs']],
        textposition = 'outside',
        textfont     = dict(color='#34d399', size=9),
        hovertemplate= '<b>Paid — Avg Installs</b><br>Installs: %{y:,.0f}<extra></extra>'
    ))

    # BAR 3 — Free Apps: Avg Revenue (Right Y-axis) — Purple
    fig.add_trace(go.Bar(
        name         = 'Free — Avg Revenue ($)',
        x            = x_fr,
        y            = free_df['Avg_Revenue'].tolist(),
        width        = 0.18,
        marker       = dict(color='#a78bfa', opacity=0.85,
                            line=dict(color='#a78bfa', width=1)),
        yaxis        = 'y2',
        text         = [f"${v/1000:.1f}K" if v >= 1000 else f"${v:.0f}"
                        for v in free_df['Avg_Revenue']],
        textposition = 'outside',
        textfont     = dict(color='#a78bfa', size=9),
        hovertemplate= '<b>Free — Avg Revenue</b><br>Revenue: $%{y:,.2f}<extra></extra>'
    ))

    # BAR 4 — Paid Apps: Avg Revenue (Right Y-axis) — Gold
    fig.add_trace(go.Bar(
        name         = 'Paid — Avg Revenue ($)',
        x            = x_pr,
        y            = paid_df['Avg_Revenue'].tolist(),
        width        = 0.18,
        marker       = dict(color='#fbbf24', opacity=0.85,
                            line=dict(color='#fbbf24', width=1)),
        yaxis        = 'y2',
        text         = [f"${v/1000:.1f}K" if v >= 1000 else f"${v:.0f}"
                        for v in paid_df['Avg_Revenue']],
        textposition = 'outside',
        textfont     = dict(color='#fbbf24', size=9),
        hovertemplate= '<b>Paid — Avg Revenue</b><br>Revenue: $%{y:,.2f}<extra></extra>'
    ))

    fig.update_layout(
        title = dict(
            text    = ('📊 Avg Installs vs Avg Revenue — Free vs Paid Apps (Top 3 Categories)<br>'
                       '<sup>Filters: Installs ≥ 10K | Revenue ≥ $10K | Android > 4.0 | '
                       'Size > 15MB | Content: Everyone | App Name ≤ 30 chars | ⏱ 1PM–2PM IST</sup>'),
            font    = dict(size=14, color='#e8eaf0'),
            x       = 0.5,
            xanchor = 'center'
        ),
        paper_bgcolor = '#0f1117',
        plot_bgcolor  = '#161b26',
        barmode       = 'overlay',

        xaxis = dict(
            tickvals  = x,
            ticktext  = [c.replace('_',' ').title() for c in top3_cats],
            tickfont  = dict(color='#8892a4', size=12),
            gridcolor = '#252c3d',
            title     = dict(text='App Category',
                             font=dict(color='#8892a4', size=12))
        ),

        yaxis = dict(
            title     = dict(text='Average Installs',
                             font=dict(color='#4f9cf9', size=12)),
            tickfont  = dict(color='#4f9cf9', size=10),
            gridcolor = '#252c3d',
            zeroline  = False
        ),

        yaxis2 = dict(
            title      = dict(text='Average Revenue (USD $)',
                              font=dict(color='#a78bfa', size=12)),
            tickfont   = dict(color='#a78bfa', size=10),
            overlaying = 'y',
            side       = 'right',
            showgrid   = False,
            zeroline   = False
        ),

        legend = dict(
            bgcolor     = '#1e2435',
            bordercolor = '#252c3d',
            borderwidth = 1,
            font        = dict(color='#8892a4', size=11),
            orientation = 'h',
            x           = 0.5,
            xanchor     = 'center',
            y           = -0.22
        ),

        margin = dict(l=60, r=60, t=110, b=90),
        height = 580,
        font   = dict(family='Inter, sans-serif'),

        annotations = [dict(
            text      = '🔵 Blue = Free Installs &nbsp; 🟢 Green = Paid Installs &nbsp; '
                        '🟣 Purple = Free Revenue &nbsp; 🟡 Gold = Paid Revenue',
            x         = 0.5, xref='paper',
            y         = 1.02, yref='paper',
            showarrow = False,
            font      = dict(color='#8892a4', size=10),
            xanchor   = 'center'
        )]
    )

    fig.show()
    print("\n✅ Dual-axis chart rendered successfully!")
    print(f"   Categories : {top3_cats}")
    print(f"   4 bars per group : Free Installs | Paid Installs | Free Revenue | Paid Revenue")

else:
    print("=" * 58)
    print("⏱  CHART ACCESS RESTRICTED")
    print("=" * 58)
    print(f"  Chart only available  : 1:00 PM – 2:00 PM IST")
    print(f"  Current IST time      : {ist_now.strftime('%I:%M:%S %p')}")
    print(f"  Next window opens     : Today/Tomorrow at 1:00 PM IST")
    print("=" * 58)

⏱  CHART ACCESS RESTRICTED
  Chart only available  : 1:00 PM – 2:00 PM IST
  Current IST time      : 07:34:50 PM
  Next window opens     : Today/Tomorrow at 1:00 PM IST
